<a href="https://colab.research.google.com/github/Arigalan/Manejo-de-Outliers-con-Python/blob/main/Pr%C3%A1ctica_manejo_de_outliers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Importamos la tablas

In [ ]:
import pandas as pd

In [ ]:
members = pd.read_csv('members.csv')
sessions = pd.read_csv('sessions.csv')
trainers = pd.read_csv('trainers.csv')

Parte 1 — LEFT JOIN

Realiza un LEFT JOIN entre:

sessions

members

Usa:

member_id

La tabla final debe contener:

información de sesiones

nombre del miembro

edad

tipo de membresía

In [ ]:
left = pd.merge(
    sessions,
    members,
    how='left',
    on='member_id'
)

In [ ]:
sessions_members = left[['session_id', 'member_id', 'member_name', 'age', 'membership_type']]
sessions_members

,session_id,member_id,member_name,age,membership_type
0,5001,1001,Juan Perez,22,Premium
1,5002,1002,Maria Lopez,31,Basic
2,5003,1003,Carlos Ramirez,27,Premium
3,5004,1004,Ana Torres,45,Basic
4,5005,1005,Diego Flores,29,VIP
5,5006,1006,Sofia Hernandez,33,Premium
6,5007,1007,Luis Mendoza,24,Basic
7,5008,1008,Valeria Castro,38,VIP
8,5009,1009,Miguel Silva,41,Premium
9,5010,1010,Fernanda Ruiz,26,Basic


Parte 2 — RIGHT JOIN

Realiza un RIGHT JOIN entre:

members

sessions

Usa:

member_id



In [ ]:
right = pd.merge(
    members,
    sessions,
    how='right',
    on='member_id'
)
right

,member_id,member_name,age,membership_type,session_id,trainer_id,session_duration,calories_burned
0,1001,Juan Perez,22,Premium,5001,201,45,320
1,1002,Maria Lopez,31,Basic,5002,202,60,450
2,1003,Carlos Ramirez,27,Premium,5003,204,55,500
3,1004,Ana Torres,45,Basic,5004,203,40,250
4,1005,Diego Flores,29,VIP,5005,205,70,620
5,1006,Sofia Hernandez,33,Premium,5006,201,50,410
6,1007,Luis Mendoza,24,Basic,5007,202,65,480
7,1008,Valeria Castro,38,VIP,5008,204,75,700
8,1009,Miguel Silva,41,Premium,5009,205,80,760
9,1010,Fernanda Ruiz,26,Basic,5010,203,35,220


Responde:
¿El resultado fue diferente al LEFT JOIN?

R: No

¿Por qué?

R: Presenta la misma cantidad de filas y si filtramos utilizando solamente las mismas columnas que pusimos en sessions_members obtendríamos los mismos datos, solamente que en el right join se agregan más columnas.

Parte 3 — INNER JOIN

Realiza un INNER JOIN entre:

sessions

trainers

Usa:

trainer_id

In [ ]:
sessions_trainers = pd.merge(
    sessions,
    trainers,
    how='inner',
    on='trainer_id'
)
sessions_trainers

,session_id,member_id,trainer_id,session_duration,calories_burned,trainer_name,specialty
0,5001,1001,201,45,320,Ricardo Diaz,Cardio
1,5002,1002,202,60,450,Gabriela Ortiz,Strength
2,5003,1003,204,55,500,Camila Navarro,Crossfit
3,5004,1004,203,40,250,Andres Vega,Yoga
4,5005,1005,205,70,620,Daniel Herrera,HIIT
5,5006,1006,201,50,410,Ricardo Diaz,Cardio
6,5007,1007,202,65,480,Gabriela Ortiz,Strength
7,5008,1008,204,75,700,Camila Navarro,Crossfit
8,5009,1009,205,80,760,Daniel Herrera,HIIT
9,5010,1010,203,35,220,Andres Vega,Yoga


Parte 4 — Primer GroupBy (Antes de limpiar)

Ejercicio 1

Obtén el promedio de calorías quemadas por entrenador.

In [ ]:
sessions_trainers.groupby('trainer_name').agg({
    'calories_burned': 'mean'
})

,calories_burned
trainer_name,
Andres Vega,235.0
Camila Navarro,600.0
Daniel Herrera,1570.0
Gabriela Ortiz,405.0
Ricardo Diaz,840.0


Ejercicio 2

Obtén la duración promedio de sesiones por tipo de membresía.

In [ ]:
right.groupby('membership_type').agg({
    'session_duration': 'mean'
})

,session_duration
membership_type,
Basic,50.0
Premium,67.5
VIP,97.5


Ejercicio 3

Obtén la cantidad total de sesiones por miembro.
Ordena los resultados de mayor a menor.

In [ ]:
sessions_members['Cantidad_Sesiones'] = sessions_members.groupby('member_name')['session_id'].transform('count')
sessions_members[['member_id', 'member_name', 'Cantidad_Sesiones']].sort_values('Cantidad_Sesiones', ascending=False)

/tmp/ipykernel_7250/2673758891.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sessions_members['Cantidad_Sesiones'] = sessions_members.groupby('member_name')['session_id'].transform('count')


,member_id,member_name,Cantidad_Sesiones
0,1001,Juan Perez,2
2,1003,Carlos Ramirez,2
16,1009,Miguel Silva,2
4,1005,Diego Flores,2
8,1009,Miguel Silva,2
7,1008,Valeria Castro,2
14,1005,Diego Flores,2
13,1003,Carlos Ramirez,2
12,1001,Juan Perez,2
11,1012,Lucia Gomez,2


Ejercicio 4

Obtén el valor máximo de calorías quemadas por entrenador.

In [ ]:
sessions_trainers.groupby('trainer_name').agg({
    'calories_burned': 'max'
})

,calories_burned
trainer_name,
Andres Vega,250
Camila Navarro,850
Daniel Herrera,3500
Gabriela Ortiz,510
Ricardo Diaz,2200


Parte 5 — Detección de Outliers usando IQR

Trabaja sobre la columna:

calories_burned

Paso 1 — Obtener Cuartiles

Calcula:

Q1

Q3

Usa:

quantile ()

In [ ]:
Q1 = sessions_trainers['calories_burned'].quantile(0.25)
Q3 = sessions_trainers['calories_burned'].quantile(0.75)

Paso 2 — Calcular IQR

IQR = Q3 − Q1

In [ ]:
IQR = Q3 - Q1

Paso 3 — Calcular Límites

lower_limit = Q1 − 1,5(IQR)

upper_limit = Q3 + 1,5(IQR)

In [ ]:
lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

Paso 4 — Detectar Outliers

Encuentra todas las filas donde:

calories_burned <lower_limit

o

calories_burned >upper_limit

In [ ]:
sessions_trainers_outliers = sessions_trainers[(sessions_trainers['calories_burned'] < lower_limit) & (sessions_trainers['calories_burned'] > upper_limit)]
sessions_trainers_outliers.shape[0]

0

Paso 5 — Limpiar Dataset

Crea un nuevo DataFrame llamado:

sessions_clean

eliminando los outliers detectados.

In [ ]:
sessions_clean = sessions_trainers[(sessions_trainers['calories_burned'] >= lower_limit) & (sessions_trainers['calories_burned'] <= upper_limit)]
sessions_clean

,session_id,member_id,trainer_id,session_duration,calories_burned,trainer_name,specialty
0,5001,1001,201,45,320,Ricardo Diaz,Cardio
1,5002,1002,202,60,450,Gabriela Ortiz,Strength
2,5003,1003,204,55,500,Camila Navarro,Crossfit
3,5004,1004,203,40,250,Andres Vega,Yoga
4,5005,1005,205,70,620,Daniel Herrera,HIIT
5,5006,1006,201,50,410,Ricardo Diaz,Cardio
6,5007,1007,202,65,480,Gabriela Ortiz,Strength
7,5008,1008,204,75,700,Camila Navarro,Crossfit
8,5009,1009,205,80,760,Daniel Herrera,HIIT
9,5010,1010,203,35,220,Andres Vega,Yoga


Parte 6 — Segundo GroupBy (Después de limpiar)

Repite los siguientes ejercicios usando:

sessions_clean

Ejercicio 1

Obtén nuevamente el promedio de calorías quemadas por entrenador.

In [ ]:
sessions_clean.groupby('trainer_name').agg({
    'calories_burned': 'mean'
})

,calories_burned
trainer_name,
Andres Vega,235.000000
Camila Navarro,600.000000
Daniel Herrera,690.000000
Gabriela Ortiz,405.000000
Ricardo Diaz,386.666667


Ejercicio 2

Obtén nuevamente la duración promedio de sesiones por tipo de membresía.

In [ ]:
sessions_clean_member = pd.merge(
    sessions_clean,
    members,
    how='left',
    on='member_id'
)

In [ ]:
sessions_clean_member.groupby('membership_type').agg({
    'session_duration': 'mean'
})

,session_duration
membership_type,
Basic,50.00
Premium,60.00
VIP,58.75


Ejercicio 3

Obtén nuevamente el valor máximo de calorías quemadas por entrenador.

In [ ]:
sessions_clean.groupby('trainer_name').agg({
    'calories_burned': 'max'
})

,calories_burned
trainer_name,
Andres Vega,250
Camila Navarro,850
Daniel Herrera,760
Gabriela Ortiz,510
Ricardo Diaz,430


Parte 7 — Comparación

Compara los resultados obtenidos:

antes de eliminar outliers

después de eliminar outliers

Responde:

1. ¿Qué cambios observaste?
2. ¿Qué métricas cambiaron más?
3. ¿Por qué los outliers afectan tanto algunos promedios?
4. ¿Siempre es correcto eliminar outliers?

1. R: De las principales diferencias que vi es que en los groupby los datos como el promedio de las calorias quemadas por entrenador o el máximo de las calorías quemadas por entrenador.

2. De donde más se nota el cambio es en las calorias quemadas promedio por entrenador

3. Los outliers pueden afectar bastante si no se usa la métrica correcta, por ejemplo, cuando calculamos la media de algunos datos puede que ésta salga modificada porque estamos ponderando un dato muy grande respecto a una serie de datos promedio, lo cual desequilibra la media.

4. Pues en general depende, es muy útil si queremos analizar los datos por un comportamiento general o cuando queremos rellenar datos vacios con la media por ejemplo, pero si queremos analizar los datos naturales pues puede ser que no sea la mejor opcion.